In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (128, 128)
BATCH_SIZE = 64
EPOCHS = 5
NUM_CLASSES = 10
OUT_DIR = "results"

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

os.makedirs(OUT_DIR, exist_ok=True)
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

val_size = 10000
x_val, y_val = x_train_full[:val_size], y_train_full[:val_size]
x_train, y_train = x_train_full[val_size:], y_train_full[val_size:]

y_train = y_train.squeeze()
y_val = y_val.squeeze()
y_test = y_test.squeeze()
# ////////////////////////////////////////////////////////////////////////////////////////////////////
def format_dataset(images, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(buffer_size=10000, seed=SEED)
    ds = ds.map(
        lambda x, y: (tf.image.resize(tf.cast(x, tf.float32), IMG_SIZE), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = format_dataset(x_train, y_train, training=True)
val_ds = format_dataset(x_val, y_val, training=False)
test_ds = format_dataset(x_test, y_test, training=False)

print("Datasets ready!")
# ////////////////////////////////////////////////////////////////////////////////////////////////////
def build_custom_cnn(input_shape=(128, 128, 3), num_classes=NUM_CLASSES):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1.0 / 255.0),

        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(256, 3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.35),
        layers.Dense(num_classes, activation="softmax")
    ], name="Custom_CNN")
    return model

cnn_model = build_custom_cnn((*IMG_SIZE, 3), NUM_CLASSES)
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()
cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)
cnn_train_time = time.time() - start_time
print(f"Custom CNN training time: {cnn_train_time:.2f} sec")
# //////////////////////////////////////////////////////////////////////////////////////////////////
def build_efficientnet(input_shape=(128, 128, 3), num_classes=NUM_CLASSES):
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = eff_preprocess(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="EfficientNetB0")

eff_model = build_efficientnet((*IMG_SIZE, 3), NUM_CLASSES)
eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()
eff_history = eff_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)
eff_train_time = time.time() - start_time
print(f"EfficientNetB0 training time: {eff_train_time:.2f} sec")
# //////////////////////////////////////////////////////////////////////////////////////////////////////
def evaluate_model(model, dataset, y_true, train_time_sec):
    test_loss, _ = model.evaluate(dataset, verbose=0)
    y_probs = model.predict(dataset, verbose=0)
    y_pred = np.argmax(y_probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    params = model.count_params()

    return {
        "Model": model.name,
        "Test Accuracy": round(float(acc), 4),
        "Test Loss": round(float(test_loss), 4),
        "Macro F1": round(float(macro_f1), 4),
        "Parameters": int(params),
        "Train Time (sec)": round(float(train_time_sec), 2)
    }, y_pred

cnn_metrics, cnn_preds = evaluate_model(cnn_model, test_ds, y_test, cnn_train_time)
eff_metrics, eff_preds = evaluate_model(eff_model, test_ds, y_test, eff_train_time)

comparison_df = pd.DataFrame([cnn_metrics, eff_metrics])
comparison_df.to_csv(f"{OUT_DIR}/comparison_results.csv", index=False)
display(comparison_df)

print("\n--- Custom CNN Classification Report ---")
print(classification_report(y_test, cnn_preds, target_names=CLASS_NAMES, digits=4))

print("\n--- EfficientNetB0 Classification Report ---")
print(classification_report(y_test, eff_preds, target_names=CLASS_NAMES, digits=4))
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(cnn_history.history["accuracy"], label="Train Acc")
axes[0, 0].plot(cnn_history.history["val_accuracy"], label="Val Acc")
axes[0, 0].set_title("Custom CNN - Accuracy")
axes[0, 0].legend()

axes[0, 1].plot(cnn_history.history["loss"], label="Train Loss")
axes[0, 1].plot(cnn_history.history["val_loss"], label="Val Loss")
axes[0, 1].set_title("Custom CNN - Loss")
axes[0, 1].legend()

axes[1, 0].plot(eff_history.history["accuracy"], label="Train Acc")
axes[1, 0].plot(eff_history.history["val_accuracy"], label="Val Acc")
axes[1, 0].set_title("EfficientNetB0 - Accuracy")
axes[1, 0].legend()

axes[1, 1].plot(eff_history.history["loss"], label="Train Loss")
axes[1, 1].plot(eff_history.history["val_loss"], label="Val Loss")
axes[1, 1].set_title("EfficientNetB0 - Loss")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.show()
# ///////////////////////////////////////////////////////////////////////////////////////////
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, preds, title in zip(
    axes,
    [cnn_preds, eff_preds],
    ["Custom CNN Confusion Matrix", "EfficientNetB0 Confusion Matrix"]
):
    cm = confusion_matrix(y_test, preds)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrices.png", dpi=150)
plt.show()
# ///////////////////////////////////////////////////////////////////////////////////////////////////////////////

IMG_PATH = "cat.jpg"

img = tf.keras.utils.load_img(IMG_PATH, target_size=IMG_SIZE)
img_array = tf.keras.utils.img_to_array(img)
img_batch = np.expand_dims(img_array, axis=0)

cnn_probs = cnn_model.predict(img_batch, verbose=0)[0]
cnn_idx = int(np.argmax(cnn_probs))
cnn_label = CLASS_NAMES[cnn_idx]
cnn_conf = float(cnn_probs[cnn_idx]) * 100

eff_probs = eff_model.predict(img_batch, verbose=0)[0]
eff_idx = int(np.argmax(eff_probs))
eff_label = CLASS_NAMES[eff_idx]
eff_conf = float(eff_probs[eff_idx]) * 100

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"CNN: {cnn_label} ({cnn_conf:.1f}%)\n"
    f"EfficientNetB0: {eff_label} ({eff_conf:.1f}%)"
)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cat_prediction.png", dpi=150)
plt.show()

print(f"Custom CNN prediction      -> {cnn_label} ({cnn_conf:.2f}%)")
print(f"EfficientNetB0 prediction -> {eff_label} ({eff_conf:.2f}%)")